# Word-Level Next-Word Prediction Using LSTM

## 1. Project Introduction

This project develops a word-level next-word prediction model using a Long Short-Term Memory (LSTM) neural network.

The model learns word patterns from a technical writing corpus and predicts the next word given a sequence of previous words.

## 2. Project Objective

The objective is to develop and evaluate an LSTM-based language model capable of predicting the next word from a given sequence of words.

The project also implements multi-word text generation using the trained model.

## 3. Dataset Description

The dataset is the `Technical-Writing.pdf` document stored in `data/raw/`.

The PDF will be converted into text and prepared for word-level language modeling.

Images and page numbers are not required for this task and will be excluded during preprocessing.

## 4. Import Libraries

The required libraries are imported for PDF extraction, text processing, data analysis, visualization, and LSTM model development.

In [ ]:
# PDF and file handling
import pymupdf
from pathlib import Path

# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Natural Language Processing
import nltk

# Machine learning utilities
from sklearn.model_selection import train_test_split

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# General utilities
import re
import string
import random

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)

## 5. Define Project Paths

Project paths are defined using `pathlib` to access the dataset and project directories consistently.

In [ ]:
# Project root
PROJECT_ROOT = Path.cwd().parent

# Project directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

# Dataset path
PDF_PATH = RAW_DATA_DIR / "Technical-Writing.pdf"

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())

## 6. Load Dataset

The Technical Writing PDF is loaded using PyMuPDF for text extraction.

In [ ]:
# Load the PDF document
pdf_document = pymupdf.open(PDF_PATH)

print("PDF loaded successfully.")
print("Number of pages:", len(pdf_document))

## 7. Inspect Raw Text

A sample of the extracted text is displayed to examine its structure before preprocessing.

In [ ]:
# Inspect the first five pages
for page_number in range(min(5, len(pdf_document))):
    page = pdf_document[page_number]
    text = page.get_text("text")

    print("=" * 80)
    print(f"PAGE {page_number + 1}")
    print("=" * 80)
    print(text[:3000])
    print()

## 8. Analyze Raw Text Extraction

The extracted text is analyzed to understand its size, page coverage, and common extraction artifacts before preprocessing.

In [ ]:
# Extract text from all pages
raw_pages = []

for page in pdf_document:
    raw_pages.append(page.get_text("text"))

# Combine all extracted pages
raw_text = "\n".join(raw_pages)

# Basic statistics
total_characters = len(raw_text)
total_words = len(raw_text.split())
non_empty_pages = sum(bool(page.strip()) for page in raw_pages)

print("Total pages:", len(raw_pages))
print("Pages containing text:", non_empty_pages)
print("Total characters:", total_characters)
print("Total words:", total_words)

## 9. Inspect Page-Level Text Statistics

Page-level statistics are calculated to identify pages with unusually little or large amounts of extracted text.

In [ ]:
# Calculate characters and words per page
page_statistics = pd.DataFrame({
    "page": range(1, len(raw_pages) + 1),
    "characters": [len(page) for page in raw_pages],
    "words": [len(page.split()) for page in raw_pages]
})

page_statistics.head()

In [ ]:
page_statistics[["characters", "words"]].describe()

## 10. Text Distribution by Page

The distribution of extracted words across pages is visualized to identify unusually short or long pages.

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    page_statistics["page"],
    page_statistics["words"]
)

plt.title("Number of Extracted Words per Page", fontsize=16)
plt.xlabel("Page Number", fontsize=12)
plt.ylabel("Word Count", fontsize=12)

plt.tight_layout()
plt.show()

## 11. Inspect Beginning and Ending Text

The beginning and ending portions of the extracted corpus are inspected to identify metadata, copyright information, references, or other content that may not be suitable for language-model training.

In [ ]:
print("FIRST 3000 CHARACTERS")
print("=" * 80)
print(raw_text[:3000])

print("\n\nLAST 3000 CHARACTERS")
print("=" * 80)
print(raw_text[-3000:])

## 12. Text Cleaning

The extracted text is cleaned to remove PDF-related artifacts while preserving the natural language required for next-word prediction.

In [ ]:
# Create a working copy of the extracted text
clean_text = raw_text

# Normalize line endings
clean_text = clean_text.replace("\r\n", "\n").replace("\r", "\n")

# Remove common control characters while preserving
# newline and tab characters
clean_text = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]", " ", clean_text)

# Normalize multiple spaces
clean_text = re.sub(r"[ \t]+", " ", clean_text)

# Normalize excessive blank lines
clean_text = re.sub(r"\n{3,}", "\n\n", clean_text)

# Remove leading/trailing whitespace
clean_text = clean_text.strip()

print("Cleaning completed.")
print("Characters before cleaning:", len(raw_text))
print("Characters after cleaning:", len(clean_text))

## 13. Inspect Cleaned Text

A sample of the cleaned corpus is displayed to verify that useful text has been preserved and unwanted extraction artifacts have been reduced.

In [ ]:
print(clean_text[:5000])

## 14. Analyze Common Text Artifacts

Frequent short lines are examined to identify repeated headers, footers, page numbers, or other PDF elements.

In [ ]:
# Extract individual non-empty lines
lines = [
    line.strip()
    for line in clean_text.splitlines()
    if line.strip()
]

# Count repeated lines
line_counts = pd.Series(lines).value_counts()

# Display the most frequent lines
line_counts.head(30)

## 15. Identify Numeric-Only Lines

Numeric-only lines are examined because isolated numbers in PDF extraction may represent page numbers.

In [ ]:
numeric_lines = [
    line for line in lines
    if re.fullmatch(r"\d+", line)
]

print("Number of numeric-only lines:", len(numeric_lines))
print("Sample:", numeric_lines[:30])

## 16. Remove Isolated Page Numbers

Lines containing only numeric values are removed because they do not contribute to the language model.

In [ ]:
# Remove lines containing only numbers
clean_lines = [
    line for line in lines
    if not re.fullmatch(r"\d+", line)
]

clean_text = "\n".join(clean_lines)

print("Text updated after removing numeric-only lines.")
print("Remaining lines:", len(clean_lines))

## 17. Inspect Updated Corpus

The corpus is inspected again after removing isolated numeric lines.

In [ ]:
print(clean_text[:5000])

## 18. Analyze Sentence Structure

The cleaned corpus is analyzed at the sentence level to understand its structure before tokenization.

In [ ]:
# Split the corpus into sentences using common sentence-ending punctuation
sentences = re.split(r'(?<=[.!?])\s+', clean_text)

# Remove empty sentences
sentences = [
    sentence.strip()
    for sentence in sentences
    if sentence.strip()
]

print("Total sentences:", len(sentences))
print("\nSample sentences:\n")

for i, sentence in enumerate(sentences[:10], start=1):
    print(f"{i}. {sentence}")

## 19. Analyze Sentence Length

Sentence lengths are analyzed to understand the distribution of words in the corpus.

In [ ]:
sentence_lengths = [
    len(sentence.split())
    for sentence in sentences
]

sentence_length_df = pd.DataFrame({
    "sentence_length": sentence_lengths
})

sentence_length_df.describe()

## 20. Sentence Length Distribution

The distribution of sentence lengths is visualized to understand the typical size of sentences in the corpus.

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    sentence_lengths,
    bins=40
)

plt.title("Sentence Length Distribution", fontsize=16)
plt.xlabel("Number of Words", fontsize=12)
plt.ylabel("Number of Sentences", fontsize=12)

plt.tight_layout()
plt.show()

## 21. Inspect Long Sentences

The longest extracted sentences are inspected to identify possible PDF extraction problems.

In [ ]:
# Display the 10 longest sentences
longest_sentences = sorted(
    sentences,
    key=lambda x: len(x.split()),
    reverse=True
)

for i, sentence in enumerate(longest_sentences[:10], start=1):
    print(f"\n{i}. Word count: {len(sentence.split())}")
    print(sentence[:1000])

## 22. Inspect Short Sentences

Very short sentences are inspected to determine whether they represent valid text or PDF artifacts.

In [ ]:
# Display sentences containing one to three words
short_sentences = [
    sentence
    for sentence in sentences
    if 1 <= len(sentence.split()) <= 3
]

print("Number of short sentences:", len(short_sentences))

for sentence in short_sentences[:30]:
    print("-", sentence)

## 23. Prepare Sentence Corpus

The cleaned text is converted into a sentence-level corpus for subsequent tokenization and sequence generation.

In [ ]:
# Keep sentences with at least 3 words
sentence_corpus = [
    sentence
    for sentence in sentences
    if len(sentence.split()) >= 3
]

print("Total sentences before filtering:", len(sentences))
print("Total sentences after filtering:", len(sentence_corpus))

## 24. Inspect Final Sentence Corpus

A sample of the prepared sentence corpus is displayed before moving to tokenization.

In [ ]:
for i, sentence in enumerate(sentence_corpus[:20], start=1):
    print(f"{i}. {sentence}")

## 25. Corpus Statistics

The complete extracted corpus is analyzed to measure its overall size and sentence-level characteristics.

In [ ]:
# Calculate corpus statistics
total_sentences = len(sentence_corpus)
total_words = sum(len(sentence.split()) for sentence in sentence_corpus)
total_characters = len(clean_text)

print("Corpus Statistics")
print("-" * 40)
print("Pages:", len(pdf_document))
print("Characters:", total_characters)
print("Sentences:", total_sentences)
print("Words:", total_words)

## 26. Sentence Length Statistics

The average, minimum, and maximum sentence lengths are calculated for the complete corpus.

In [ ]:
sentence_lengths = np.array([
    len(sentence.split())
    for sentence in sentence_corpus
])

print("Minimum sentence length:", sentence_lengths.min())
print("Maximum sentence length:", sentence_lengths.max())
print("Average sentence length:", round(sentence_lengths.mean(), 2))
print("Median sentence length:", np.median(sentence_lengths))

## 27. Full-Corpus Word Frequency

Word frequencies are calculated to understand the most common words in the complete corpus.

In [ ]:
# Extract words from the complete corpus
all_words = re.findall(r"\b[\w']+\b", clean_text.lower())

word_frequency = pd.Series(all_words).value_counts()

print("Total word tokens:", len(all_words))
print("Unique words:", len(word_frequency))

word_frequency.head(20)

## 28. Vocabulary Statistics

The vocabulary size and word-frequency distribution are examined before tokenization.

In [ ]:
vocabulary_size = len(word_frequency)

print("Vocabulary size:", vocabulary_size)
print("Total word tokens:", len(all_words))

## 29. Most Frequent Words

The most frequent words in the corpus are visualized to understand the vocabulary distribution.

In [ ]:
top_words = word_frequency.head(20)

plt.figure(figsize=(12, 6))

top_words.sort_values().plot(kind="barh")

plt.title("Top 20 Most Frequent Words", fontsize=16)
plt.xlabel("Frequency", fontsize=12)
plt.ylabel("Word", fontsize=12)

plt.tight_layout()
plt.show()

## 30. Analyze Repeated Lines

Frequently occurring lines are analyzed across the complete PDF to identify possible repeated headers, footers, or other document elements.

In [ ]:
# Count lines across the complete cleaned corpus
line_counts = pd.Series(
    line.strip()
    for line in clean_text.splitlines()
    if line.strip()
).value_counts()

# Display frequently repeated lines
repeated_lines = line_counts[line_counts >= 10]

print("Number of lines appearing at least 10 times:",
      len(repeated_lines))

repeated_lines.head(30)

## 31. Analyze Numeric-Only Lines

Numeric-only lines are counted across the complete document to determine whether they are likely to represent page numbers or other numerical content.

In [ ]:
numeric_line_counts = pd.Series(
    line.strip()
    for line in clean_text.splitlines()
    if re.fullmatch(r"\d+", line.strip())
).value_counts()

print("Total numeric-only lines:",
      numeric_line_counts.sum())

print("\nMost common numeric-only lines:")
print(numeric_line_counts.head(20))

## 32. Analyze Very Short Lines

Short lines are examined to distinguish meaningful headings or content from possible PDF extraction artifacts.

In [ ]:
short_lines = [
    line.strip()
    for line in clean_text.splitlines()
    if line.strip() and len(line.split()) <= 3
]

print("Total short lines:", len(short_lines))

print("\nSample short lines:")
for line in short_lines[:50]:
    print("-", line)

## 33. Identify Repeated Document Elements

Repeated lines are reviewed to identify document elements that should not become part of the language-model corpus.

In [ ]:
# Display frequently repeated lines
for line, count in repeated_lines.head(50).items():
    print(f"{count:>4}  {line}")

## 34. Identify Page-Number Patterns

Numeric-only lines are reviewed to determine whether they represent page numbers or meaningful numerical content.

In [ ]:
# Display numeric-only lines and their frequencies
for value, count in numeric_line_counts.head(50).items():
    print(f"{count:>4}  {value}")

## 35. Create Final Cleaned Corpus

The identified PDF artifacts are removed while preserving meaningful textual content, punctuation, and sentence structure.

In [ ]:
# Start again from the extracted pages
processed_pages = []

for page_text in raw_pages:
    lines = page_text.splitlines()

    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Remove isolated numeric page numbers
        if re.fullmatch(r"\d+", line):
            continue

        # Normalize whitespace
        line = re.sub(r"\s+", " ", line)

        cleaned_lines.append(line)

    processed_pages.append("\n".join(cleaned_lines))

# Combine all processed pages
final_text = "\n".join(processed_pages)

# Normalize excessive whitespace
final_text = re.sub(r"\n{3,}", "\n\n", final_text)
final_text = final_text.strip()

print("Final corpus created.")
print("Characters:", len(final_text))
print("Words:", len(final_text.split()))

## 36. Inspect Final Corpus

A sample of the final corpus is displayed to verify that the preprocessing preserved meaningful text.

In [ ]:
print(final_text[:5000])

## 37. Compare Corpus Before and After Cleaning

The corpus size before and after preprocessing is compared to measure the effect of the cleaning process.

In [ ]:
comparison = pd.DataFrame({
    "Stage": ["Raw Text", "Final Text"],
    "Characters": [
        len(raw_text),
        len(final_text)
    ],
    "Words": [
        len(raw_text.split()),
        len(final_text.split())
    ]
})

comparison

## 38. Save Processed Corpus

The cleaned corpus is saved in the processed data directory for reproducibility and future model development.

In [ ]:
# Create processed data directory if it does not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save the final corpus
PROCESSED_TEXT_PATH = PROCESSED_DATA_DIR / "technical_writing_corpus.txt"

with open(PROCESSED_TEXT_PATH, "w", encoding="utf-8") as file:
    file.write(final_text)

print("Processed corpus saved successfully.")
print("Path:", PROCESSED_TEXT_PATH)
print("File exists:", PROCESSED_TEXT_PATH.exists())

## 39. Load the Processed Corpus

The cleaned corpus is loaded from the processed data directory so that the remaining preprocessing steps operate on the saved dataset rather than directly on the original PDF.

In [ ]:
# Load the processed corpus

with open(PROCESSED_TEXT_PATH, "r", encoding="utf-8") as file:
    corpus_text = file.read()

print("Processed corpus loaded successfully.")
print("Characters:", len(corpus_text))
print("Words:", len(corpus_text.split()))

## 40. Verify the Processed Corpus

The beginning and end of the processed corpus are inspected to verify that the saved text can be loaded correctly and that the document content has not been unintentionally truncated.

In [ ]:
print("First 2000 characters:")
print("=" * 70)
print(corpus_text[:2000])

print("\n\nLast 2000 characters:")
print("=" * 70)
print(corpus_text[-2000:])

## 41. Sentence Segmentation

The cleaned corpus is divided into individual sentences. Sentence boundaries are important because the language model should learn natural word sequences rather than sequences created by arbitrary breaks in the document.

In [ ]:
# Sentence segmentation

sentence_corpus = re.split(
    r"(?<=[.!?])\s+",
    corpus_text
)

# Remove empty sentences
sentence_corpus = [
    sentence.strip()
    for sentence in sentence_corpus
    if sentence.strip()
]

print("Total sentences:", len(sentence_corpus))

print("\nSample sentences:")
for sentence in sentence_corpus[:10]:
    print("-", sentence)

## 42. Inspect Sentence Segmentation

A sample of the segmented sentences is inspected to verify that sentence boundaries have been detected correctly.

In [ ]:
for i, sentence in enumerate(sentence_corpus[:20], start=1):
    print(f"{i:02d}. {sentence}")

## 43. Sentence Length Distribution

The distribution of sentence lengths is analyzed to understand how many words typically occur in each sentence.

In [ ]:
sentence_lengths = np.array([
    len(sentence.split())
    for sentence in sentence_corpus
])

print("Minimum sentence length:", sentence_lengths.min())
print("Maximum sentence length:", sentence_lengths.max())
print("Average sentence length:", round(sentence_lengths.mean(), 2))
print("Median sentence length:", np.median(sentence_lengths))

## 44. Sentence Length Distribution

The sentence-length distribution is visualized to identify unusually short or long sentences and to understand the structure of the corpus.

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(sentence_lengths, bins=50)

plt.title("Sentence Length Distribution", fontsize=16)
plt.xlabel("Number of Words", fontsize=12)
plt.ylabel("Number of Sentences", fontsize=12)

plt.tight_layout()
plt.show()

## 45. Create a Word-Level Representation

The corpus is converted into a word-level representation because the proposed language model predicts the next word based on the sequence of previously observed words.

In [ ]:
# Extract words from the complete corpus

all_words = re.findall(
    r"\b[\w']+\b",
    corpus_text.lower()
)

print("Total word tokens:", len(all_words))
print("Unique words:", len(set(all_words)))

print("\nSample words:")
print(all_words[:50])

## 46. Analyze the Word Vocabulary

The vocabulary is examined before tokenization to understand the number of unique words available to the language model.

In [ ]:
# Build vocabulary frequency

word_frequency = pd.Series(all_words).value_counts()

vocabulary = sorted(word_frequency.index)

print("Total word tokens:", len(all_words))
print("Vocabulary size:", len(vocabulary))

print("\nFirst 50 vocabulary items:")
print(vocabulary[:50])

## 47. Analyze Rare Words

Word-frequency counts are used to identify rare vocabulary items. This helps determine whether vocabulary reduction should be considered before training the model.

In [ ]:
# Frequency distribution of words

frequency_distribution = word_frequency.value_counts().sort_index()

print("Words appearing once:",
      (word_frequency == 1).sum())

print("Words appearing two times:",
      (word_frequency == 2).sum())

print("Words appearing five times or fewer:",
      (word_frequency <= 5).sum())

print("Words appearing ten times or fewer:",
      (word_frequency <= 10).sum())

## 48. Word-Frequency Distribution

The distribution of word frequencies is visualized to examine how common and rare words are distributed throughout the corpus.

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(
    word_frequency.values,
    bins=50
)

plt.title("Word Frequency Distribution", fontsize=16)
plt.xlabel("Word Frequency", fontsize=12)
plt.ylabel("Number of Words", fontsize=12)

plt.tight_layout()
plt.show()

## 49. Vocabulary Strategy

The vocabulary strategy is selected based on the characteristics of the complete corpus.

For this educational next-word prediction project, the initial model will preserve the observed vocabulary rather than aggressively removing rare technical words.

This allows the model to learn terminology that may be uncommon in general English but important within technical writing.

In [ ]:
# Preserve the complete observed vocabulary

selected_vocabulary = vocabulary

print("Selected vocabulary size:",
      len(selected_vocabulary))

## 50. Add Special Tokens

Special tokens are introduced to represent information that is not part of the original vocabulary.

The initial vocabulary will contain:

- `<PAD>` for padding sequences
- `<UNK>` for unknown words
- `<START>` to indicate the beginning of a sentence
- `<END>` to indicate the end of a sentence

These tokens allow the model and preprocessing pipeline to handle sequence boundaries and vocabulary mismatches consistently.

In [ ]:
SPECIAL_TOKENS = [
    "<PAD>",
    "<UNK>",
    "<START>",
    "<END>"
]

final_vocabulary = SPECIAL_TOKENS + selected_vocabulary

print("Vocabulary including special tokens:",
      len(final_vocabulary))

print("\nSpecial tokens:")
for token in SPECIAL_TOKENS:
    print(token)

## 51. Create Word-to-Index Mapping

Each vocabulary word is assigned a unique integer index. This mapping converts textual words into numerical values that can be processed by the neural network.

In [ ]:
word_to_index = {
    word: index
    for index, word in enumerate(final_vocabulary)
}

print("Sample word-to-index mappings:")

for word in final_vocabulary[:20]:
    print(f"{word:15} -> {word_to_index[word]}")

## 52. Create Index-to-Word Mapping

A reverse mapping is created so that numerical predictions generated by the model can be converted back into their corresponding words.

In [ ]:
index_to_word = {
    index: word
    for word, index in word_to_index.items()
}

print("Sample index-to-word mappings:")

for index in range(20):
    print(f"{index:5} -> {index_to_word[index]}")

## 53. Verify the Vocabulary Mappings

The word-to-index and index-to-word mappings are verified to ensure that the vocabulary conversion process is reversible.

In [ ]:
# Test a few words

test_words = [
    "<PAD>",
    "<UNK>",
    "<START>",
    "<END>",
    "technical",
    "writing"
]

for word in test_words:
    index = word_to_index.get(word)

    if index is not None:
        reconstructed_word = index_to_word[index]

        print(
            f"{word} -> {index} -> {reconstructed_word}"
        )

## 54. Convert Sentences into Token Sequences

Each sentence is converted from words into their corresponding integer indices.

Sentence boundaries are represented using `<START>` and `<END>` tokens so that the model can learn where sequences begin and end.

In [ ]:
tokenized_sentences = []

for sentence in sentence_corpus:

    words = re.findall(
        r"\b[\w']+\b",
        sentence.lower()
    )

    tokens = ["<START>"] + words + ["<END>"]

    token_ids = [
        word_to_index.get(word, word_to_index["<UNK>"])
        for word in tokens
    ]

    tokenized_sentences.append(token_ids)

print("Number of tokenized sentences:",
      len(tokenized_sentences))

print("\nFirst tokenized sentence:")
print(tokenized_sentences[0])

## 55. Inspect Tokenized Sentences

Several tokenized sentences are converted back into words for inspection to verify that the numerical representation correctly represents the original text.

In [ ]:
for i, token_ids in enumerate(tokenized_sentences[:10], start=1):

    words = [
        index_to_word[token_id]
        for token_id in token_ids
    ]

    print(f"{i:02d}.", " ".join(words))

## 56. Check Unknown Tokens

The occurrence of `<UNK>` tokens is measured to determine whether words are being lost during the conversion from text to numerical sequences.

In [ ]:
unk_index = word_to_index["<UNK>"]

total_tokens = sum(
    len(sequence)
    for sequence in tokenized_sentences
)

unknown_tokens = sum(
    token_id == unk_index
    for sequence in tokenized_sentences
    for token_id in sequence
)

print("Total tokens:", total_tokens)
print("Unknown tokens:", unknown_tokens)

print(
    "Unknown-token percentage:",
    round(
        (unknown_tokens / total_tokens) * 100,
        4
    ),
    "%"
)

## 57. Analyze Tokenized Sequence Lengths

The lengths of the tokenized sentences are analyzed to determine an appropriate sequence length for the LSTM training process.

In [ ]:
token_sequence_lengths = np.array([
    len(sequence)
    for sequence in tokenized_sentences
])

print("Minimum sequence length:",
      token_sequence_lengths.min())

print("Maximum sequence length:",
      token_sequence_lengths.max())

print("Average sequence length:",
      round(token_sequence_lengths.mean(), 2))

print("Median sequence length:",
      np.median(token_sequence_lengths))

## 58. Token Sequence Length Distribution

The distribution of tokenized sentence lengths is visualized to determine a practical sequence length for model training.

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(
    token_sequence_lengths,
    bins=50
)

plt.title(
    "Tokenized Sentence Length Distribution",
    fontsize=16
)

plt.xlabel(
    "Number of Tokens",
    fontsize=12
)

plt.ylabel(
    "Number of Sentences",
    fontsize=12
)

plt.tight_layout()
plt.show()

## 59. Determine the LSTM Sequence Length

A practical sequence length is selected for the language model based on the observed sentence-length distribution.

The sequence length determines how many previous words the LSTM receives when learning to predict the next word.

This value should provide enough context for prediction while keeping the training process computationally manageable.

In [ ]:
# Initial sequence length for the language model

SEQUENCE_LENGTH = 20

print("Selected sequence length:",
      SEQUENCE_LENGTH)

## 60. Prepare Training Sequences

The tokenized sentences are converted into overlapping input-target sequences.

For each sequence, the input contains a fixed number of previous words and the target is the next word.

For example, with a sequence length of 5:

Input:
`The student is writing a`

Target:
`technical`

This transforms the corpus into supervised training examples for the LSTM language model.

In [ ]:
input_sequences = []
target_words = []

for token_ids in tokenized_sentences:

    # Skip sentences that are too short
    if len(token_ids) <= SEQUENCE_LENGTH:
        continue

    for i in range(SEQUENCE_LENGTH, len(token_ids)):

        input_sequence = token_ids[i-SEQUENCE_LENGTH:i]
        target_word = token_ids[i]

        input_sequences.append(input_sequence)
        target_words.append(target_word)

print("Total training sequences:",
      len(input_sequences))

print("Input sequence length:",
      len(input_sequences[0]))

print("Number of target words:",
      len(target_words))

## 61. Inspect Training Sequence Examples

Several generated input-target pairs are converted back into words and displayed to verify that the sequence-generation process correctly represents the next-word prediction task.

In [ ]:
for i in range(min(10, len(input_sequences))):

    input_words = [
        index_to_word[token_id]
        for token_id in input_sequences[i]
    ]

    target_word = index_to_word[target_words[i]]

    print("Input :", " ".join(input_words))
    print("Target:", target_word)
    print("-" * 70)

## 62. Convert Training Data into NumPy Arrays

The generated input sequences and target words are converted into NumPy arrays so they can be efficiently processed during model training.

In [ ]:
X = np.array(
    input_sequences,
    dtype=np.int32
)

y = np.array(
    target_words,
    dtype=np.int32
)

print("X shape:", X.shape)
print("y shape:", y.shape)

## 63. Verify Input and Target Alignment

The relationship between each input sequence and its corresponding target word is verified before the dataset is divided for model training.

In [ ]:
# Verify several input-target pairs

for i in range(5):

    input_text = " ".join(
        index_to_word[token_id]
        for token_id in X[i]
    )

    target_text = index_to_word[y[i]]

    print(f"Example {i + 1}")
    print("Input :", input_text)
    print("Target:", target_text)
    print()

## 64. Determine Dataset Size

The total number of generated training examples is calculated before splitting the dataset into training, validation, and test subsets.

In [ ]:
dataset_size = len(X)

print("Total examples:", dataset_size)
print("Input examples:", X.shape[0])
print("Target examples:", y.shape[0])

## 65. Prepare the Dataset Split

The dataset is divided into training, validation, and test portions while preserving sentence-level separation.

This prevents nearly identical sequences originating from the same sentence from appearing in different subsets.

The training set is used to learn the language model, the validation set is used to monitor model performance during development, and the test set is reserved for final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

# Split sentence indices first
sentence_indices = np.arange(len(tokenized_sentences))

train_indices, temp_indices = train_test_split(
    sentence_indices,
    test_size=0.20,
    random_state=42
)

validation_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42
)

print("Training sentences:", len(train_indices))
print("Validation sentences:", len(validation_indices))
print("Test sentences:", len(test_indices))

## 66. Generate Training, Validation, and Test Sequences

Training sequences are generated separately from the training, validation, and test sentences.

This ensures that the same source sentence cannot contribute sequences to multiple dataset splits.

In [ ]:
def create_sequences_from_sentences(
    sentences,
    sequence_length
):
    """
    Convert tokenized sentences into
    input-target training sequences.
    """

    inputs = []
    targets = []

    for token_ids in sentences:

        if len(token_ids) <= sequence_length:
            continue

        for i in range(
            sequence_length,
            len(token_ids)
        ):

            inputs.append(
                token_ids[i-sequence_length:i]
            )

            targets.append(
                token_ids[i]
            )

    return (
        np.array(inputs, dtype=np.int32),
        np.array(targets, dtype=np.int32)
    )

## 67. Create the Three Dataset Splits

The previously separated sentences are converted into numerical input-target sequences for training, validation, and testing.

In [ ]:
train_sentences = [
    tokenized_sentences[i]
    for i in train_indices
]

validation_sentences = [
    tokenized_sentences[i]
    for i in validation_indices
]

test_sentences = [
    tokenized_sentences[i]
    for i in test_indices
]

X_train, y_train = create_sequences_from_sentences(
    train_sentences,
    SEQUENCE_LENGTH
)

X_validation, y_validation = create_sequences_from_sentences(
    validation_sentences,
    SEQUENCE_LENGTH
)

X_test, y_test = create_sequences_from_sentences(
    test_sentences,
    SEQUENCE_LENGTH
)

print("Training:")
print("X:", X_train.shape)
print("y:", y_train.shape)

print("\nValidation:")
print("X:", X_validation.shape)
print("y:", y_validation.shape)

print("\nTest:")
print("X:", X_test.shape)
print("y:", y_test.shape)

## 68. Verify Dataset Separation

The three datasets are inspected to confirm that they contain usable sequences and that their dimensions are consistent.

In [ ]:
print("Dataset Summary")
print("=" * 60)

print(
    f"Training examples   : {len(X_train):,}"
)

print(
    f"Validation examples : {len(X_validation):,}"
)

print(
    f"Test examples       : {len(X_test):,}"
)

print(
    f"Sequence length     : {SEQUENCE_LENGTH}"
)

print(
    f"Vocabulary size     : {len(final_vocabulary):,}"
)

## 69. Save the Preprocessed Dataset

The prepared training, validation, and test datasets are saved so that model development can be performed without repeating the PDF extraction and preprocessing stages.

In [ ]:
# Create processed data directory
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save datasets
np.save(
    PROCESSED_DATA_DIR / "X_train.npy",
    X_train
)

np.save(
    PROCESSED_DATA_DIR / "y_train.npy",
    y_train
)

np.save(
    PROCESSED_DATA_DIR / "X_validation.npy",
    X_validation
)

np.save(
    PROCESSED_DATA_DIR / "y_validation.npy",
    y_validation
)

np.save(
    PROCESSED_DATA_DIR / "X_test.npy",
    X_test
)

np.save(
    PROCESSED_DATA_DIR / "y_test.npy",
    y_test
)

print("Preprocessed datasets saved successfully.")

## 70. Save the Vocabulary

The word-to-index and index-to-word mappings are saved so that the same vocabulary can be used during model training, evaluation, and real-world next-word prediction.

In [ ]:
import json

VOCAB_PATH = PROCESSED_DATA_DIR / "vocabulary.json"

vocabulary_data = {
    "word_to_index": word_to_index,
    "index_to_word": {
        str(index): word
        for index, word in index_to_word.items()
    }
}

with open(
    VOCAB_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        vocabulary_data,
        file,
        ensure_ascii=False,
        indent=4
    )

print("Vocabulary saved successfully.")
print("Path:", VOCAB_PATH)

## 71. Final Preprocessing Summary

The complete text preprocessing pipeline is summarized before moving to model construction.

The workflow has transformed the original PDF into numerical sequences suitable for training an LSTM-based next-word prediction model.

In [ ]:
print("=" * 70)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 70)

print(f"Corpus characters       : {len(corpus_text):,}")
print(f"Corpus words            : {len(corpus_text.split()):,}")
print(f"Sentences               : {len(sentence_corpus):,}")
print(f"Vocabulary size         : {len(final_vocabulary):,}")
print(f"Sequence length         : {SEQUENCE_LENGTH}")

print("\nDataset sizes")
print("-" * 70)

print(f"Training examples       : {len(X_train):,}")
print(f"Validation examples     : {len(X_validation):,}")
print(f"Test examples           : {len(X_test):,}")

print("\nInput shape")
print("-" * 70)

print(f"X_train                 : {X_train.shape}")
print(f"X_validation            : {X_validation.shape}")
print(f"X_test                  : {X_test.shape}")

print("\nPreprocessing completed successfully.")

## 72. Introduction to the LSTM Next-Word Prediction Model

The preprocessed corpus is now converted into numerical sequences that can be used to train a neural language model.

An LSTM (Long Short-Term Memory) network is used because it can learn relationships between words across a sequence and use previously observed context to predict the next word.

The model receives a fixed-length sequence of words and predicts the word that should come next.

For example:

Input:
`technical writing is an`

Target:
`important`

The trained model will later be used recursively to generate multiple words.

## 73. Import Deep Learning Libraries

The required TensorFlow and Keras components are imported for constructing, training, evaluating, and saving the LSTM language model.

In [ ]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

print("TensorFlow version:", tf.__version__)

## 74. Check Available Hardware

The available computational device is checked before training the neural network.

In [ ]:
# Check available devices

devices = tf.config.list_physical_devices()

print("Available devices:")
for device in devices:
    print("-", device)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("\nGPU detected.")
    print("Number of GPUs:", len(gpus))
else:
    print("\nNo GPU detected. Training will use the CPU.")

## 75. Define Model Configuration

The main hyperparameters of the language model are defined in one place so that the experiment can be reproduced and modified easily.

The embedding dimension controls the size of the word representation, while the LSTM units determine the capacity of the recurrent layer.

In [ ]:
# Model configuration

VOCAB_SIZE = len(final_vocabulary)

EMBEDDING_DIM = 128
LSTM_UNITS = 256

DROPOUT_RATE = 0.2

BATCH_SIZE = 128
EPOCHS = 30

LEARNING_RATE = 0.001

print("Model Configuration")
print("-" * 50)
print("Vocabulary size :", VOCAB_SIZE)
print("Sequence length :", SEQUENCE_LENGTH)
print("Embedding dim   :", EMBEDDING_DIM)
print("LSTM units      :", LSTM_UNITS)
print("Dropout         :", DROPOUT_RATE)
print("Batch size      :", BATCH_SIZE)
print("Epochs          :", EPOCHS)
print("Learning rate   :", LEARNING_RATE)

## 76. Model Output Size

The model must produce a probability distribution across the entire vocabulary.

For every input sequence, the model therefore produces one probability for every word in the vocabulary.

In [ ]:
print("Number of possible output words:", VOCAB_SIZE)

## 77. Build the LSTM Architecture

The language model consists of three main components:

1. Embedding layer
2. LSTM layer
3. Dense output layer

The embedding layer converts integer word IDs into dense vector representations.

The LSTM learns sequential relationships between those representations.

The final Dense layer produces a probability distribution over the vocabulary.

In [ ]:
model = Sequential([
    
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=SEQUENCE_LENGTH
    ),
    
    LSTM(
        LSTM_UNITS
    ),
    
    Dropout(
        DROPOUT_RATE
    ),
    
    Dense(
        VOCAB_SIZE,
        activation="softmax"
    )
])

print("LSTM model created successfully.")

## 78. Model Summary

The architecture and number of trainable parameters are displayed to verify that the model has been constructed correctly.

In [ ]:
model.summary()

## 79. Compile the Model

The model is compiled using the Adam optimizer and sparse categorical cross-entropy loss.

Sparse categorical cross-entropy is appropriate because the target for each training example is represented by a single integer corresponding to the correct next word.

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

## 80. Verify the Compiled Model

A small batch from the training dataset is passed through the model before training begins.

This confirms that the input dimensions and output dimensions are compatible.

In [ ]:
# Take a small sample

sample_X = X_train[:2]

sample_predictions = model.predict(
    sample_X,
    verbose=0
)

print("Input shape:", sample_X.shape)
print("Prediction shape:", sample_predictions.shape)

## 81. Check Prediction Probabilities

The model's initial predictions are inspected to verify that the output represents a probability distribution over the vocabulary.

In [ ]:
sample_prediction = sample_predictions[0]

print("Probability sum:",
      sample_prediction.sum())

print("Minimum probability:",
      sample_prediction.min())

print("Maximum probability:",
      sample_prediction.max())

## 82. Identify the Initial Predicted Word

The word with the highest initial probability is identified.

Because the model has not yet been trained, this prediction is not expected to be meaningful. It is used only to verify the prediction pipeline.

In [ ]:
predicted_index = np.argmax(
    sample_prediction
)

predicted_word = index_to_word[
    predicted_index
]

print("Initial predicted word:", predicted_word)

## 83. Configure Early Stopping

Early stopping is used to prevent unnecessary training once validation performance stops improving.

The model weights from the best validation epoch are restored after training.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

## 84. Configure Model Checkpoint

A checkpoint is created to save the model whenever validation loss improves.

This provides a persistent copy of the best-performing model during training.

In [ ]:
CHECKPOINT_PATH = MODEL_DIR / "best_lstm_model.keras"

model_checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

## 85. Configure Learning-Rate Reduction

If validation loss stops improving, the learning rate is reduced.

This allows the optimizer to make smaller parameter updates when the model approaches a better solution.

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

## 86. Train the LSTM Model

The LSTM language model is trained using the training dataset.

The validation dataset is evaluated after each epoch to monitor whether the model is learning patterns that generalize beyond the training data.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    
    validation_data=(
        X_validation,
        y_validation
    ),
    
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    
    callbacks=[
        early_stopping,
        model_checkpoint,
        reduce_lr
    ],
    
    verbose=1
)

## 88. Inspect Training History

The training history contains the loss and accuracy recorded during each training epoch.

These values are used to understand the learning behavior of the model.

In [ ]:
history_data = pd.DataFrame(
    history.history
)

history_data

## 89. Training and Validation Loss

Training and validation loss are plotted to examine how the model's prediction error changes during training.

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title(
    "Training and Validation Loss",
    fontsize=16
)

plt.xlabel(
    "Epoch",
    fontsize=12
)

plt.ylabel(
    "Loss",
    fontsize=12
)

plt.legend()

plt.tight_layout()
plt.show()

## 90. Training and Validation Accuracy

Training and validation accuracy are visualized to examine how often the model predicts the correct next word.

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title(
    "Training and Validation Accuracy",
    fontsize=16
)

plt.xlabel(
    "Epoch",
    fontsize=12
)

plt.ylabel(
    "Accuracy",
    fontsize=12
)

plt.legend()

plt.tight_layout()
plt.show()

## 91. Determine Best Training Epoch

The epoch with the lowest validation loss is identified to determine when the model achieved its best validation performance during training.

In [ ]:
best_epoch = np.argmin(
    history.history["val_loss"]
) + 1

best_val_loss = min(
    history.history["val_loss"]
)

best_val_accuracy = history.history[
    "val_accuracy"
][best_epoch - 1]

print("Best epoch:", best_epoch)
print("Best validation loss:",
      round(best_val_loss, 4))

print("Validation accuracy at best epoch:",
      round(best_val_accuracy, 4))

## 92. Load the Best Saved Model

The best checkpoint is loaded so that subsequent evaluation and text generation use the model associated with the best validation loss.

In [ ]:
# Path to the best model saved during training
BEST_MODEL_PATH = MODEL_DIR / "best_lstm_model.keras"

# Load the best model
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)

print("=" * 70)
print("BEST MODEL LOADED")
print("=" * 70)
print(f"Model path: {BEST_MODEL_PATH}")
print("Model loaded successfully.")

## 93. Verify the Best Model

Before saving the final model artifact, we verify that the loaded model has the expected architecture and can successfully process input data.

This confirms that the saved checkpoint can be loaded correctly and is ready for use outside the training notebook.

In [91]:
# Display the model architecture
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (2, 20, 128)           │     1,351,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (2, 256)               │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (2, 256)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (2, 10556)             │     2,712,892 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,374,902 (51.02 MB)

 Trainable params: 4,458,300 (17.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,916,602 (34.01 MB)

In [92]:
# Verify the model using one sample from the test dataset

sample_input = X_test[:1]

sample_prediction = best_model.predict(
    sample_input,
    verbose=0
)

predicted_index = int(np.argmax(sample_prediction[0]))
actual_index = int(y_test[0])

print("=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print(f"Input shape:       {sample_input.shape}")
print(f"Output shape:      {sample_prediction.shape}")
print(f"Predicted index:   {predicted_index}")
print(f"Predicted word:    {index_to_word[predicted_index]}")
print(f"Actual index:      {actual_index}")
print(f"Actual word:       {index_to_word[actual_index]}")

MODEL VERIFICATION
Input shape:       (1, 20)
Output shape:      (1, 10556)
Predicted index:   3
Predicted word:    <END>
Actual index:      8189
Actual word:       right


## 94. Save the Final Best Model

The verified best model is now saved as the final model artifact.

This file will be used later by the separate inference notebook and by the web application. The model does not need to be retrained when making predictions.

In [93]:
# Final model path
FINAL_MODEL_PATH = MODEL_DIR / "technical_writing_lstm_best.keras"

# Save the best model
best_model.save(FINAL_MODEL_PATH)

print("=" * 70)
print("FINAL MODEL SAVED")
print("=" * 70)
print(f"Saved to: {FINAL_MODEL_PATH}")
print(f"File exists: {FINAL_MODEL_PATH.exists()}")

FINAL MODEL SAVED
Saved to: c:\Projects_Tracking\Word_Predictor_LSTM\models\technical_writing_lstm_best.keras
File exists: True


## 96. Save the Model Configuration

The inference system must know the important configuration values used during training.

These include the sequence length, vocabulary size, embedding dimension, LSTM units, dropout rate, and special tokens.

Saving these values prevents the prediction notebook and web application from relying on manually entered values.

In [ ]:
# Path for model configuration
CONFIG_PATH = MODEL_DIR / "model_config.json"

# Configuration used by the trained model
model_config = {
    "sequence_length": SEQUENCE_LENGTH,
    "vocabulary_size": VOCAB_SIZE,
    "embedding_dim": EMBEDDING_DIM,
    "lstm_units": LSTM_UNITS,
    "dropout_rate": DROPOUT_RATE,
    "special_tokens": {
        "padding": "<PAD>",
        "unknown": "<UNK>",
        "start": "<START>",
        "end": "<END>"
    }
}

# Save configuration
with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(
        model_config,
        file,
        ensure_ascii=False,
        indent=4
    )

print("=" * 70)
print("MODEL CONFIGURATION SAVED")
print("=" * 70)
print(f"Saved to: {CONFIG_PATH}")

## 98. Final Training Notebook Summary

The complete LSTM training pipeline has now been finished.

The Technical Writing PDF was processed into a text corpus, cleaned and prepared for next-word prediction, converted into numerical sequences, and used to train an LSTM language model.

The best-performing model was selected during training and saved as the final model.

The required supporting files are also available for the next stage of the project.

### Final Project Artifacts

- `technical_writing_lstm_best.keras` — the trained best LSTM model
- `vocabulary.json` — the vocabulary and word-index mappings
- `model_config.json` — the configuration required for model inference

The training notebook is now complete.

No prediction or text-generation code will be added to this notebook. The next stage will be developed in a separate notebook dedicated to loading the saved model and performing next-word prediction.

In [99]:
print("=" * 80)
print("TRAINING NOTEBOOK COMPLETED")
print("=" * 80)

print("\nFinal Model:")
print(FINAL_MODEL_PATH)

print("\nVocabulary:")
print(VOCABULARY_PATH)

print("\nModel Configuration:")
print(CONFIG_PATH)

print("\n" + "=" * 80)
print("READY FOR THE NEXT-PREDICTION NOTEBOOK")
print("=" * 80)

TRAINING NOTEBOOK COMPLETED

Final Model:
c:\Projects_Tracking\Word_Predictor_LSTM\models\technical_writing_lstm_best.keras

Vocabulary:
c:\Projects_Tracking\Word_Predictor_LSTM\models\vocabulary.json

Model Configuration:
c:\Projects_Tracking\Word_Predictor_LSTM\models\model_config.json

READY FOR THE NEXT-PREDICTION NOTEBOOK
